# 11 — The fixed starter kit **with** the DeepSeek news signals  *(branch `exp/11-kit-news`)*

Experiment 10 again — same data, factors, 4 DQN members, seeds, 320,000 steps, slippage 7e-7, 70/15/15 split —
with one change: the state gets the organisers' DeepSeek **sentiment** and **risk** scores of the week's news
(2 + 8 + **2** inputs). The price-only run (`starter_kit/runs/shipped`, notebook 10) is the comparison.

**How the scores are mapped onto each second** (`starter_kit/news_signals.py`, settings fixed in advance):

| | kit / organisers' csv (and E9) | here |
|---|---|---|
| articles | all 279 | 232 that mention bitcoin / crypto in the title or opening (drops e.g. EUR/USD, Walmart, casinos) |
| timing | publication time | publication time (or the later Yahoo URL time) **+ 60 s** latency, causal |
| value at second *t* | the latest article's score, held until the next | **decayed, confidence-weighted average** of all articles so far (half-life 1 h) |
| no recent news | the last score stays for hours | fades back to 0 = the train-period average |
| scale | 1–5 | centred on the train mean, ÷ train 99th percentile of \|s\|, clipped to ±1 |

Runs on its own runtime, in parallel with notebook 10: both only **read** `starter_kit/prepared/` and write
to different `runs/` folders. **Pick an A100** (replay buffer ~28 GB at 4096 envs).

**Inputs** (Drive, never committed): `thesis_rl/starter_kit/prepared/` from notebook 10, plus
`BTC_1sec_signals.npy` (+ `.json`) in the same folder — made locally with
`python kit_pipeline.py news --csv BTC_1sec.csv --news news_train.csv --data prepared`, or built here if
`contest/data/BTC_1sec.csv` and `contest/data/news_train.csv` are on Drive.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/11-kit-news"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > A100 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "11_starter_kit_news")
sys.path.insert(0, f"{EXP}/starter_kit")             # the kit's flat modules (erl_agent, trade_simulator, ...)
import kit_pipeline as kp

## Inputs: notebook 10's prepared data and factors, and the per-second news signals

In [ ]:
import json
from pathlib import Path
KIT = Path(DRIVE_ROOT) / "starter_kit"
DATA = KIT / "prepared"
assert (DATA / "BTC_1sec_predict.npy").exists(), f"run notebook 10 first: no RNN factors in {DATA}"
if not (DATA / "BTC_1sec_signals.npy").exists():
    src = Path(DRIVE_ROOT) / "contest" / "data"
    assert (src / "BTC_1sec.csv").exists() and (src / "news_train.csv").exists(), \
        f"upload BTC_1sec_signals.npy (+ .json) to {DATA}, or BTC_1sec.csv and news_train.csv to {src}"
    kp.news(str(src / "BTC_1sec.csv"), str(src / "news_train.csv"), str(DATA))
print(json.dumps(json.load(open(DATA / "BTC_1sec_signals.json")), indent=1))

## Ensemble with news: 4 members × 320,000 steps

Each member is saved when it finishes, so after a disconnect **Run all** again: finished members are skipped
(an unfinished member restarts). About 65–75 min per member on an A100, ~4.5 h in all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
RUN = KIT / "runs" / "news_decay"                    # slippage 7e-7, as the price-only run in runs/shipped
kp.ensemble(str(DATA), str(RUN), gpu_id=0, break_step=int(32e4), num_sims="auto", slippage=7e-7, seed=0, signals=True)

## Results: news vs price only (validation selects, test scored once)

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
BASE = KIT / "runs" / "shipped"
for name, run in (("price only", BASE), ("with news", RUN)):
    for m in sorted((run / "ensemble_models").iterdir()):
        s = json.loads((m / "selection.json").read_text())
        print(f"{name:10s} {m.name:18s} best validation ${s['best_val_return']:8.2f} per episode at step {s['best_step']:,}")
for split in ("val", "test"):
    kp.evaluate(str(DATA), str(RUN), split=split, gpu_id=0)
cols = ["cum_return_pct", "max_drawdown_pct", "sharpe_per_step", "trades", "episodes_positive", "episode_return_pct_median"]
rows = []
for split in ("val", "test"):
    for name, run in (("price only", BASE), ("with news", RUN)):
        f = run / f"eval_{split}" / "summary.json"
        if not f.exists():
            print("missing", f); continue
        for c in json.loads(f.read_text())["curves"]:
            if c["curve"] == "sim" or (c["curve"] == "bh_1btc" and name == "price only"):
                rows.append(dict(split=split, run=name if c["curve"] == "sim" else "buy & hold 1 BTC", **{k: c[k] for k in cols}))
table = pd.DataFrame(rows)
table.to_csv(RUN / "comparison.csv", index=False)
display(table)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for run, name, lab in ((BASE, "sim", "price only"), (RUN, "sim", "with news"), (BASE, "bh_1btc", "buy & hold 1 BTC")):
    e = np.load(run / "eval_test" / f"equity_{name}.npy")
    axes[0].plot((e / 1e6 - 1) * 100, lw=1.3, label=lab)
axes[0].set(title="Test: chained 79-min episodes, $1M, slippage 7e-7", xlabel="2-second steps", ylabel="cumulative return (%)")
axes[0].axhline(0, color="grey", lw=0.8); axes[0].legend()
for m in sorted((RUN / "ensemble_models").iterdir()):
    c = np.array(json.loads((m / "selection.json").read_text())["val_curve"])
    axes[1].plot(c[:, 0], c[:, 1], lw=1.3, label=m.name)
axes[1].set(title="With news: validation return per episode during training", xlabel="training step", ylabel="$ per episode")
axes[1].axhline(0, color="grey", lw=0.8); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(RUN / "11_starter_kit_news.png", dpi=150); plt.show()

## Shut the runtime down

Everything is on Drive. Releasing the GPU stops compute units being used while the runtime sits idle
(set `SHUTDOWN = False` to keep it).

In [ ]:
SHUTDOWN = True
if SHUTDOWN and "google.colab" in sys.modules:
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()